# 26. Large-Scale Spectrum Occupancy Learning via Tensor Decomposition and LSTM Networks

Implements the **tensor CP (CANDECOMP/PARAFAC) + LSTM** method from:
**"Large-Scale Spectrum Occupancy Learning via Tensor Decomposition and LSTM Networks"** (IEEE 2020).

## Paper (adapted to 72h→24h)
- **Tensor:** 3-way spectrum tensor (channels × time × samples). We build **(N_channels, 96, n_train)** with 96 = 72 (input) + 24 (target) per sample.
- **CP decomposition:** X ≈ ⟨A, B, C⟩ with A (N_channels×R), B (96×R), C (n_train×R). Fit with ALS.
- **LSTM on time factor:** Train LSTM to map B[:72,:] → B[72:,:] (predict the "future" 24 steps of the temporal factor from the first 72).
- **Test:** For new input (72, N_channels), solve for sample factor **c_new** from 72h input via least squares; predict 24h as A @ diag(c_new) @ B_pred.T where B_pred = LSTM(B[:72,:]).

## Same setup as 10–25
Data: work_dir/final, multi-channel aligned 72h→24h. Metrics: MAE, RMSE, MASE. Naive baseline. Same visuals.

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
print(f'TensorFlow: {tf.__version__}')


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus, 'GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU enabled: {len(gpus)} device(s)')
        with tf.device('/GPU:0'): _ = tf.constant(1)
        print('GPU ready.')
    except RuntimeError as e: print('GPU config:', e)
else: print('No GPU. On Apple Silicon: pip install tensorflow-metal')
USE_GPU = len(gpus) > 0
num_cores = os.cpu_count() or 4
tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)


## Data loading: multi-channel aligned (same as notebook 24/25)

In [ ]:
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"
if not final_dir.exists() or not training_dir.exists() or not testing_dir.exists():
    raise FileNotFoundError("work_dir/final/training and testing not found")
class_options = sorted([d.name for d in training_dir.iterdir() if d.is_dir()])
LOOKBACK = 72
FORECAST_HORIZON = 24

def load_data_for_band(band_name: str, split: str):
    split_dir = final_dir / split / band_name
    if not split_dir.exists(): return pd.DataFrame()
    dfs = []
    for p in sorted(split_dir.glob("final_*.parquet")):
        try: dfs.append(pd.read_parquet(p))
        except Exception as e: print(f"Error loading {p}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def build_multichannel_sequences(train_dfs, test_dfs, bands, lookback=72, horizon=24):
    merged_tr = None
    for band in bands:
        df = train_dfs[band]
        if df.empty: continue
        df = df.sort_values(["date", "hour"]).drop_duplicates(["date", "hour"])
        ths = sorted(df["threshold_dbm"].unique())
        if len(ths) > 1: df = df[df["threshold_dbm"] == ths[0]]
        au = df.groupby(["date", "hour"])["au_pct"].mean().reset_index().rename(columns={"au_pct": band})
        merged_tr = au if merged_tr is None else pd.merge(merged_tr, au, on=["date", "hour"], how="inner")
    if merged_tr is None or len(merged_tr) < lookback + horizon:
        return None, None, None, None, []
    band_cols = [b for b in bands if b in merged_tr.columns]
    N = len(band_cols)
    mat_tr = merged_tr[band_cols].values.astype(np.float32)
    merged_te = None
    for band in bands:
        df = test_dfs[band]
        if df.empty: continue
        df = df.sort_values(["date", "hour"])
        ths = sorted(df["threshold_dbm"].unique())
        if len(ths) > 1: df = df[df["threshold_dbm"] == ths[0]]
        au = df.groupby(["date", "hour"])["au_pct"].mean().reset_index().rename(columns={"au_pct": band})
        merged_te = au if merged_te is None else pd.merge(merged_te, au, on=["date", "hour"], how="inner")
    for c in band_cols:
        if c not in merged_te.columns: merged_te[c] = np.nan
    merged_te = merged_te[band_cols].ffill().fillna(0)
    mat_te = merged_te.values.astype(np.float32) if len(merged_te) > 0 else np.zeros((0, N))
    X_tr, y_tr = [], []
    for i in range(len(mat_tr) - lookback - horizon + 1):
        X_tr.append(mat_tr[i:i+lookback])
        y_tr.append(mat_tr[i+lookback:i+lookback+horizon])
    X_te, y_te = [], []
    for d in range(len(mat_te) // horizon):
        if d == 0: inp = mat_tr[-lookback:] if len(mat_tr) >= lookback else np.vstack([np.zeros((lookback - len(mat_tr), N)), mat_tr])
        else:
            h = max(0, lookback - d * horizon)
            inp = np.vstack([mat_tr[-h:], mat_te[:d*horizon]]) if h > 0 else mat_te[d*horizon - lookback:d*horizon]
        tgt = mat_te[d*horizon:(d+1)*horizon]
        if len(inp) == lookback and len(tgt) == horizon:
            X_te.append(inp)
            y_te.append(tgt)
    if not X_tr or not X_te:
        return None, None, None, None, band_cols
    return np.array(X_tr), np.array(y_tr), np.array(X_te), np.array(y_te), band_cols


In [ ]:
train_data_by_band = {}
test_data_by_band = {}
for band in class_options:
    tr = load_data_for_band(band, "training")
    te = load_data_for_band(band, "testing")
    if not tr.empty and not te.empty:
        train_data_by_band[band] = tr
        test_data_by_band[band] = te
bands_used = [b for b in class_options if b in train_data_by_band and b in test_data_by_band]
out = build_multichannel_sequences(train_data_by_band, test_data_by_band, bands_used, LOOKBACK, FORECAST_HORIZON)
if out[0] is None:
    raise ValueError("Multi-channel sequences could not be built.")
X_train, y_train, X_test, y_test, band_cols = out
N_CHANNELS = len(band_cols)
print(f"Train: {X_train.shape}, Test: {X_test.shape}, Channels: {N_CHANNELS}")


## Scale and build 3-way tensor (N_channels, 96, n_train)

In [ ]:
scaler_x = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))
X_train_flat = X_train.reshape(-1, N_CHANNELS)
y_train_flat = y_train.reshape(-1, N_CHANNELS)
scaler_x.fit(X_train_flat)
scaler_y.fit(y_train_flat)
X_train_s = scaler_x.transform(X_train_flat).reshape(X_train.shape)
y_train_s = scaler_y.transform(y_train_flat).reshape(y_train.shape)
X_test_s = scaler_x.transform(X_test.reshape(-1, N_CHANNELS)).reshape(X_test.shape)

# Tensor: (N_channels, 96, n_train). Each slice [:, :, i] = [X_train[i].T | y_train[i].T]
n_train = X_train_s.shape[0]
tensor_list = []
for i in range(n_train):
    # (72, N) and (24, N) -> stack as (96, N) then store as (N, 96)
    block = np.concatenate([X_train_s[i], y_train_s[i]], axis=0).T  # (N_channels, 96)
    tensor_list.append(block)
X_tensor = np.stack(tensor_list, axis=2).astype(np.float64)  # (N_channels, 96, n_train)
print(f"Tensor shape: {X_tensor.shape}")


## CP-ALS decomposition (rank R)

In [ ]:
def khatri_rao(A, B):
    """Khatri-Rao (column-wise Kronecker). A (I,R), B (J,R) -> (I*J, R)."""
    R = A.shape[1]
    cols = []
    for r in range(R):
        cols.append(np.outer(A[:, r], B[:, r]).flatten('F'))
    return np.column_stack(cols)

def cp_als(X, rank, max_iter=100, tol=1e-6):
    """CP-ALS for 3-way tensor X (I, J, K). Returns A (I,R), B (J,R), C (K,R)."""
    I, J, K = X.shape
    R = rank
    np.random.seed(42)
    A = np.random.randn(I, R).astype(np.float64)
    B = np.random.randn(J, R).astype(np.float64)
    C = np.random.randn(K, R).astype(np.float64)
    for it in range(max_iter):
        # Mode-1: A = X(1) (C ⊙ B) ( (C'C) * (B'B) )^{-1}
        X1 = X.reshape(I, -1)
        CB = khatri_rao(C, B)
        M = (C.T @ C) * (B.T @ B)
        M = np.maximum(M, 1e-10)
        A = (X1 @ CB) @ np.linalg.inv(M)
        # Mode-2: B = X(2) (C ⊙ A) ( (C'C) * (A'A) )^{-1}
        X2 = X.transpose(1, 0, 2).reshape(J, -1)
        CA = khatri_rao(C, A)
        M = (C.T @ C) * (A.T @ A)
        M = np.maximum(M, 1e-10)
        B = (X2 @ CA) @ np.linalg.inv(M)
        # Mode-3: C = X(3) (B ⊙ A) ( (B'B) * (A'A) )^{-1}
        X3 = X.transpose(2, 0, 1).reshape(K, -1)
        BA = khatri_rao(B, A)
        M = (B.T @ B) * (A.T @ A)
        M = np.maximum(M, 1e-10)
        C = (X3 @ BA) @ np.linalg.inv(M)
    return A, B, C

RANK = min(8, N_CHANNELS, 96, n_train)
RANK = max(1, RANK)
A_cp, B_cp, C_cp = cp_als(X_tensor, RANK)
print(f"CP rank R={RANK}. A: {A_cp.shape}, B: {B_cp.shape}, C: {C_cp.shape}")


## LSTM on temporal factor B: B[:72,:] -> B[72:,:]

In [ ]:
B_input = B_cp[:LOOKBACK, :].astype(np.float32)   # (72, R)
B_target = B_cp[LOOKBACK:, :].astype(np.float32)  # (24, R)

lstm_in = B_input[np.newaxis, :, :]  # (1, 72, R)
lstm_out = B_target[np.newaxis, :, :]  # (1, 24, R)

model_b = keras.Sequential([
    layers.LSTM(32, activation='tanh', return_sequences=False, input_shape=(LOOKBACK, RANK)),
    layers.Dense(FORECAST_HORIZON * RANK, activation='linear'),
    layers.Reshape((FORECAST_HORIZON, RANK))
])
model_b.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
model_b.fit(lstm_in, lstm_out, epochs=100, verbose=0)

B_pred_lstm = model_b.predict(lstm_in, verbose=0)[0]  # (24, R)
B_pred = B_pred_lstm.astype(np.float64)
print(f"LSTM trained: B[:72] -> B[72:]. B_pred shape: {B_pred.shape}")


## Test-time: solve c_new from 72h input, predict 24h

In [ ]:
B72 = B_cp[:LOOKBACK, :]   # (72, R)
BA72 = khatri_rao(A_cp, B72)  # (N_channels*72, R); vec(ch*72+t) = X[ch,t]

def predict_cp_lstm(input_72, A, B_future, BA72, scaler_y):
    """input_72 (72, N_channels). Returns (24, N_channels) in original scale."""
    vec = input_72.T.flatten('F')  # match Khatri-Rao layout (N_channels, 72)
    c_new, _, _, _ = np.linalg.lstsq(BA72, vec, rcond=None)
    pred_24 = (A @ (c_new[:, np.newaxis] * B_future.T)).T  # (24, N_channels)
    pred_24 = scaler_y.inverse_transform(pred_24)
    return np.clip(pred_24, 0, 100).astype(np.float32)

y_pred_cp = np.zeros((len(X_test), FORECAST_HORIZON, N_CHANNELS), dtype=np.float32)
for i in range(len(X_test)):
    y_pred_cp[i] = predict_cp_lstm(X_test_s[i], A_cp, B_pred, BA72, scaler_y)

def naive_predictor_multi(X, horizon):
    last = X[:, -1, :]
    return np.tile(last[:, np.newaxis, :], (1, horizon, 1))
y_pred_naive = naive_predictor_multi(X_test, FORECAST_HORIZON)
print('CP+LSTM predictions and Naive baseline computed.')


In [ ]:
def calculate_mae(y_true, y_pred):
    return mean_absolute_error(y_true.flatten(), y_pred.flatten())
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
def calculate_mase(y_true, y_pred, y_train):
    mae = np.mean(np.abs(y_true.flatten() - y_pred.flatten()))
    scale = np.mean(np.abs(np.diff(y_train.flatten()))) if len(y_train.flatten()) > 1 else 1.0
    return mae / max(scale, 1e-8)

mae_cp = calculate_mae(y_test, y_pred_cp)
rmse_cp = calculate_rmse(y_test, y_pred_cp)
mase_cp = calculate_mase(y_test, y_pred_cp, y_train)
mae_n = calculate_mae(y_test, y_pred_naive)
rmse_n = calculate_rmse(y_test, y_pred_naive)
mase_n = calculate_mase(y_test, y_pred_naive, y_train)

results_df = pd.DataFrame([
    {"Model": "CP+LSTM", "MAE": mae_cp, "RMSE": rmse_cp, "MASE": mase_cp},
    {"Model": "Naive Baseline", "MAE": mae_n, "RMSE": rmse_n, "MASE": mase_n},
])
results_df["MAE"] = results_df["MAE"].round(4)
results_df["RMSE"] = results_df["RMSE"].round(4)
results_df["MASE"] = results_df["MASE"].round(4)

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(f"Lookback: {LOOKBACK}h, Forecast: {FORECAST_HORIZON}h, Channels: {N_CHANNELS}, CP rank: {RANK}")
print(f"Test samples: {len(y_test)}")
print(results_df.to_string(index=False))
display(results_df)


## Analysis and Visualizations (same as notebook 10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MASE']
colors = ['#2ecc71', '#3498db', '#9b59b6']
for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric].values
    bars = ax.bar(results_df['Model'], vals, color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.tick_params(axis='x', rotation=15)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02 * max(vals), f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.suptitle('CP+LSTM (tensor decomposition + LSTM on B factor)', y=1.02, fontsize=12)
plt.show()

naive_mae = results_df[results_df['Model'] == 'Naive Baseline']['MAE'].values[0]
naive_rmse = results_df[results_df['Model'] == 'Naive Baseline']['RMSE'].values[0]
naive_mase = results_df[results_df['Model'] == 'Naive Baseline']['MASE'].values[0]
improvement = results_df[results_df['Model'] != 'Naive Baseline'].copy()
improvement['MAE_imp_%'] = (1 - improvement['MAE'] / naive_mae) * 100
improvement['RMSE_imp_%'] = (1 - improvement['RMSE'] / naive_rmse) * 100
improvement['MASE_imp_%'] = (1 - improvement['MASE'] / naive_mase) * 100
print('Improvement over Naive Baseline (%):')
display(improvement[['Model', 'MAE_imp_%', 'RMSE_imp_%', 'MASE_imp_%']].round(2))
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(improvement))
w = 0.25
ax.bar(x - w, improvement['MAE_imp_%'], w, label='MAE', color='#2ecc71')
ax.bar(x, improvement['RMSE_imp_%'], w, label='RMSE', color='#3498db')
ax.bar(x + w, improvement['MASE_imp_%'], w, label='MASE', color='#9b59b6')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(improvement['Model'], rotation=15)
ax.set_ylabel('Improvement (%)')
ax.legend()
ax.set_title('Improvement over Naive (positive = better)')
plt.tight_layout()
plt.show()


In [ ]:
hours = np.arange(FORECAST_HORIZON)
ch_show = 0
n_show = min(3, len(y_test))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4*n_show))
if n_show == 1: axes = [axes]
for i in range(n_show):
    ax = axes[i]
    ax.plot(hours, y_test[i, :, ch_show], 'k--', linewidth=2.5, label='Actual', alpha=0.8)
    ax.plot(hours, y_pred_cp[i, :, ch_show], '-', linewidth=1.6, label='CP+LSTM')
    ax.plot(hours, y_pred_naive[i, :, ch_show], '-', linewidth=1.2, label='Naive', alpha=0.7)
    ax.set_title(f'Test sample {i+1}, channel {band_cols[ch_show]}')
    ax.set_xlabel('Hour')
    ax.set_ylabel('AU (%)')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, y_test.mean(axis=0).mean(axis=1), 'k--', linewidth=2.5, label='Actual (mean)')
ax.plot(hours, y_pred_cp.mean(axis=0).mean(axis=1), '-', linewidth=1.6, label='CP+LSTM (mean)')
ax.plot(hours, y_pred_naive.mean(axis=0).mean(axis=1), '-', linewidth=1.2, label='Naive (mean)', alpha=0.7)
ax.set_title('Mean 24h profile (avg over channels)')
ax.set_xlabel('Hour')
ax.set_ylabel('AU (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

mae_per_hour = np.abs(y_test - y_pred_cp).mean(axis=(0, 2))
mae_per_hour_n = np.abs(y_test - y_pred_naive).mean(axis=(0, 2))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours, mae_per_hour, '-o', label='CP+LSTM', markersize=4)
ax.plot(hours, mae_per_hour_n, '-o', label='Naive', markersize=4, alpha=0.7)
ax.set_title('MAE by forecast hour (avg over channels)')
ax.set_xlabel('Hour')
ax.set_ylabel('MAE (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
residuals_cp = (y_test - y_pred_cp).flatten()
residuals_naive = (y_test - y_pred_naive).flatten()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals_cp, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Count')
axes[0].set_title('Residuals: CP+LSTM')
axes[1].hist(residuals_naive, bins=50, color='#95a5a6', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals: Naive')
plt.suptitle('Residual distribution', y=1.02)
plt.tight_layout()
plt.show()
print(f'Residual mean (bias): CP+LSTM = {residuals_cp.mean():.4f}, Naive = {residuals_naive.mean():.4f}')
print(f'Residual std:         CP+LSTM = {residuals_cp.std():.4f}, Naive = {residuals_naive.std():.4f}')

best_row = results_df[results_df['Model'] == 'CP+LSTM'].iloc[0]
imp_mae = (1 - best_row['MAE'] / naive_mae) * 100
print(f"\nBest model: CP+LSTM (MAE={best_row['MAE']:.4f}, RMSE={best_row['RMSE']:.4f}, MASE={best_row['MASE']:.4f}). Improvement over Naive: MAE {imp_mae:+.1f}%.")


### Key insights

- **Paper (IEEE 2020):** 3-way spectrum tensor is decomposed with CP; LSTM predicts the temporal factor (here: future 24 steps of B from first 72).
- **Adaptation:** Tensor (N_channels, 96, n_train); test sample gets a **c_new** from 72h input via least squares; 24h = A @ diag(c_new) @ B_pred.T with B_pred from LSTM.
- **Improvement over Naive:** Positive % means CP+LSTM beats the last-value baseline.
